In [3]:
import pyterrier as pt
import pandas as pd


dataset = pt.get_dataset("msmarco_passage")
# print(f'Dataset: \n{dataset}')
index = dataset.get_index(variant="terrier_stemmed")

# Load DL-hard annotations
annotations_url = "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/annotations/query/annotations.tsv"
annotations = pd.read_csv(
    annotations_url,
    sep="\t",
    header=None,
    names=["qid", "query", "intent", "answer_type", "topic_domain", "serp_type"],
)
annotations["qid"] = annotations["qid"].astype(str)

# Load TREC DL 2019/2020 qrels. This dataset contains relevance judgements from scale 0 to 3.
dl19 = pt.get_dataset("irds:msmarco-passage/trec-dl-2019")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020")
all_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels()])
all_qrels["qid"] = all_qrels["qid"].astype(str)

# Keep only queries that have both annotations and judgements.
qrels_qids = set(all_qrels["qid"].unique())
topics = annotations[annotations["qid"].isin(qrels_qids)].reset_index(drop=True)

print(f'qrels: \n{all_qrels}')

print(f"Queries with both annotations and relevance values: {len(topics)}")
print(topics["intent"].value_counts())
print("\n")
print(topics["answer_type"].value_counts())
topics["query_length"] = topics["query"].str.len()
max_len = topics["query_length"].max()
bins = range(0, max_len + 10, 10)

topics["query_length_group"] = pd.cut(topics["query_length"], bins=bins, labels=[f"{i+1}-{i+10}" for i in bins][:-1])
print(f"topics: \n{topics}")

qrels: 
           qid    docno  label iteration
0        19335  1017759      0        Q0
1        19335  1082489      0        Q0
2        19335   109063      0        Q0
3        19335  1160863      0        Q0
4        19335  1160871      0        Q0
...        ...      ...    ...       ...
11381  1136962  8526087      0         0
11382  1136962  8537921      0         0
11383  1136962  8742482      0         0
11384  1136962   937258      1         0
11385  1136962   999215      0         0

[20646 rows x 4 columns]
Queries with both annotations and relevance values: 97
intent
description     48
list            10
quantity         9
reason           8
attribute        7
entity           5
verification     3
process          3
language         3
weather          1
Name: count, dtype: int64


answer_type
factoid              28
definition           21
long answer          16
list                 10
short description     7
comparison            5
short answer          5
guide         

In [9]:
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from itertools import combinations

def run_kruskal_pairwise(results_df, groupby_col, methods, measures):
    kruskal_rows = []
    pairwise_rows = []
    groups = results_df[groupby_col].dropna().unique()

    for method in methods:
        for measure in measures:
            subset = results_df[
                (results_df["name"] == method) & (results_df["measure"] == measure)
            ]
            group_scores = {
                g: subset[subset[groupby_col] == g]["value"].values
                for g in groups
            }
            valid_groups = [g for g, s in group_scores.items() if len(s) >= 2]
            if len(valid_groups) < 2:
                continue

            kw_stat, kw_p = kruskal(*[group_scores[g] for g in valid_groups])
            kruskal_rows.append({
                "method": method,
                "measure": measure,
                "kw_statistic": kw_stat,
                "kw_p_value": kw_p,
                "group_means": {g: group_scores[g].mean() for g in valid_groups},
            })

            for g1, g2 in combinations(valid_groups, 2):
                s1, s2 = group_scores[g1], group_scores[g2]
                stat, p = mannwhitneyu(s1, s2, alternative="two-sided")
                pairwise_rows.append({
                    "method": method,
                    "measure": measure,
                    "group_1": g1,
                    "group_2": g2,
                    "mw_statistic": stat,
                    "p_value": p,
                    "mean_group_1": s1.mean(),
                    "mean_group_2": s2.mean(),
                })

    kruskal_df = pd.DataFrame(kruskal_rows)
    pairwise_df = pd.DataFrame(pairwise_rows)

    if len(pairwise_df) > 0:
        reject, p_corr, _, _ = multipletests(pairwise_df["p_value"], method="holm")
        pairwise_df["p_corrected"] = p_corr
        pairwise_df["significant"] = reject

    return kruskal_df, pairwise_df


# Now extract methods and measures from the results dataframe
methods = results["name"].unique()
measures = results["measure"].unique()

# Run the tests
kw_len, pw_len = run_kruskal_pairwise(results, "query_length_group", methods, measures)

print("=== Kruskal-Wallis: Query Length ===")
print(kw_len.to_string(index=False))

print("\n=== Pairwise Mann-Whitney (Holm-corrected): Query Length ===")
print(pw_len.sort_values("p_corrected").to_string(index=False))

=== Kruskal-Wallis: Query Length ===
method      measure  kw_statistic  kw_p_value                                                                                               group_means
  BM25      nDCG@10      1.630874    0.442446   {'Medium (4-6)': 0.4582525752218138, 'Long (7+)': 0.5269270565879146, 'Short (≤3)': 0.5276914938699462}
  BM25    AP(rel=2)      1.685028    0.430627 {'Medium (4-6)': 0.26618480126721344, 'Long (7+)': 0.2913641471408059, 'Short (≤3)': 0.39031147403816885}
  BM25 R(rel=2)@100      0.713929    0.699798   {'Medium (4-6)': 0.5440192109416444, 'Long (7+)': 0.5076052348881452, 'Short (≤3)': 0.6030653809564929}
  BM25 RR(rel=2)@10      0.631642    0.729190    {'Medium (4-6)': 0.6009353741496598, 'Long (7+)': 0.683375850340136, 'Short (≤3)': 0.6085164835164835}
   Bo1      nDCG@10      1.449260    0.484504  {'Medium (4-6)': 0.47049292283600147, 'Long (7+)': 0.5456670161212867, 'Short (≤3)': 0.5350984007131147}
   Bo1    AP(rel=2)      1.674887    0.432816   {'M

In [7]:
# Create query length groups BEFORE running the experiment
# so the merge in the next cell actually finds the column
topics["query_word_count"] = topics["query"].str.split().str.len()

def length_group(n):
    if n <= 3:
        return "Short (≤3)"
    elif n <= 6:
        return "Medium (4-6)"
    else:
        return "Long (7+)"

topics["query_length_group"] = topics["query_word_count"].apply(length_group)
print(topics["query_length_group"].value_counts())

query_length_group
Medium (4-6)    56
Long (7+)       28
Short (≤3)      13
Name: count, dtype: int64


In [10]:
import re

def clean_query(q):
    q = re.sub(r"[^\w\s]", " ", str(q))
    q = re.sub(r"\s+", " ", q).strip()
    return q

# Clean queries (else it gives errors in PyTerrier)
topics["query"] = topics["query"].apply(clean_query)

# Initialize BM25 retriever
bm25 = pt.terrier.Retriever(index, wmodel="BM25")

# Initialize classic query expansion models
rm3 = pt.rewrite.RM3(index)
bo1 = pt.rewrite.Bo1QueryExpansion(index)

bm25_rm3 = bm25 >> rm3 >> bm25
bm25_bo1 = bm25 >> bo1 >> bm25

In [11]:
from pyterrier.measures import RR, nDCG, AP, R

results_classical = pt.Experiment(
    [bm25, bm25_rm3, bm25_bo1],
    topics[["qid", "query"]],  
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["BM25", "RM3", "Bo1"],
    perquery=True,
)

results = pd.concat([results_classical], ignore_index=True)

label_map = topics[["qid", "intent", "query_length_group", "meta_answer_type"]]
results = results.merge(label_map, on="qid", how="left")

# Overall results
print("\n=== Overall Results ===")
overall = results.groupby(["name", "measure"])["value"].mean().unstack()
print(overall)

# Results by query length
print("\n=== nDCG@10 by Query Length ===")
ndcg_by_length = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "query_length_group"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_length)


KeyError: "['meta_answer_type'] not in index"